In [ ]:
# ============================================================
# PANDAS: INGESTION, CLEANING & SLICING
# GRADED ASSESSMENT
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load the dataset as strings.
# Loading as strings prevents Pandas from silently converting
# values before we have inspected the data.
# ------------------------------------------------------------

df = pd.read_excel(
    r"C:\Users\EHU_HEALTH_TECHNICAL\Documents\Khing_FM_Data_Science\Data_Science\assets\ts-academy_datasets\cAlwUp3sSzSjtoyrfBXL_saas.xlsx",
    dtype=str
)

print("Dataset loaded successfully.")

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

## Creating a Working DataFrame for Analysis

For the analysis and data-cleaning tasks, I created a separate copy of the original dataset called `analysis_df`.

The original `df` is kept unchanged so that the raw dataset remains available for comparison and reference. The `analysis_df` DataFrame is used as a working copy where numerical columns can be converted, calculated fields can be created, and data-quality checks can be performed.

In particular:

- `seats` is converted from text to numeric values and stored as `seats_num`.
- `mrr` is converted from text to numeric values and stored as `mrr_num`.
- `revenue_per_seat` is calculated using MRR divided by the number of seats.
- Date columns are later parsed and validated.
- Suspicious, missing, or inconsistent values can be investigated without altering the original dataset.

This approach helps preserve the integrity of the raw data while providing a separate DataFrame for analysis and cleaning.

In [ ]:
# Create a working copy for analysis
analysis_df = df.copy()

# Convert seats to numeric
analysis_df['seats_num'] = pd.to_numeric(
    analysis_df['seats'],
    errors='coerce'
)

# Convert MRR to numeric
analysis_df['mrr_num'] = pd.to_numeric(
    analysis_df['mrr'],
    errors='coerce'
)

print("analysis_df created successfully.")
display(analysis_df.head())

# Q1 - PREDIC THE ALIGNMENT

Prediction
Before running the code, I predict that the result will contain the four unique index labels from both Series: Apr, Feb, Jan and Mar. Feb should be equal 210 because 200 + 10 = 210, while mar should be equal to 320 because 300 + 20 = 320. Jan and Apr should contain NaN because those labels occur in only one of the twi Series.

In [ ]:
# ============================================================
# Q1. PREDICT THE ALIGNMENT
# ============================================================

a = pd.Series({
    'jan': 100,
    'feb': 200,
    'mar': 300
})

b = pd.Series({
    'feb': 10,
    'mar': 20,
    'apr': 40
})

result = a + b

print("Series a:")
print(a)

print("\nSeries b:")
print(b)

print("\nActual result:")
print(result)

Expected actual result
apr      NaN
feb    210.0
jan      NaN
mar    320.0
dtype: float64

# Rationale

Pandas performs index alignment when operating on Series. It creates the union of the index labels and matches values using those labels rather than their physical positions. Therefore, feb and mar can be added, while jan and apr have no matching value and become NaN.

# Q2 — Series or DataFrame?

The assessment asks for the returned type and why the distinction matters.

In [ ]:
# ============================================
# Q2. SERIES OR DATAFRAME?
# ============================================

q2_results = {
    "df['mrr']": type(df['mrr']).__name__,
    "df[['mrr']]": type(df[['mrr']]).__name__,
    "df.loc[:, 'mrr']": type(df.loc[:, 'mrr']).__name__,
    "df.loc[:, ['mrr']]": type(df.loc[:, ['mrr']]).__name__,
}

for expression, result_type in q2_results.items():
    print(f"{expression} --> {result_type}")

# Result

| Expression                    | Type      | Why it matters                                      |
| ----------------------------- | --------- | --------------------------------------------------- |
| `df['mrr']`                   | Series    | One-dimensional; Series methods/indexing apply      |
| `df[['mrr']]`                 | DataFrame | Two-dimensional; can use DataFrame operations       |
| `df.loc[:, 'mrr']`            | Series    | Selecting one label returns one dimension           |
| `df.loc[:, ['mrr', 'seats']]` | DataFrame | Selecting a list of labels preserves two dimensions |

# Rationale

Selecting a single column label generally returns a Series, while selecting a list of columns returns a DataFrame. This matters when an operation expects two-dimensional input, such as some machine-learning functions or DataFrame-specific operations.

# Q3 — Four Routes to the Same Columns

The required columns are company, seats, and mrr. The assessment specifically requests different mechanisms.

In [ ]:
# ============================================================
# Q3. FOUR ROUTES TO THE SAME COLUMNS
# ============================================================

# 1. By column names
route_1 = df[['company', 'seats', 'mrr']]

# 2. By .loc
route_2 = df.loc[:, ['company', 'seats', 'mrr']]

# 3. By .iloc positions
route_3 = df.iloc[:, [1, 4, 5]]

# 4. By .filter
route_4 = df.filter(items=['company', 'seats', 'mrr'])

print("Route 1:")
display(route_1)

print("Route 2:")
display(route_2)

print("Route 3:")
display(route_3)

print("Route 4:")
display(route_4)

In [ ]:
# Verify that all four routes select the same columns

print("Route 1 columns:", route_1.columns.tolist())
print("Route 2 columns:", route_2.columns.tolist())
print("Route 3 columns:", route_3.columns.tolist())
print("Route 4 columns:", route_4.columns.tolist())

In [ ]:
# Recommended production approach

production_selection = df[['company', 'seats', 'mrr']]

display(production_selection)

# Rationale

The .iloc approach is fragile if a colleague reorders columns because it depends on column positions rather than names. The .loc label-range approach is also potentially fragile if columns are renamed or their ordering changes, because a range such as 'company':'mrr' depends on the labels and their order. For production code, I would use explicit column names with df[['company', 'seats', 'mrr']] because it communicates the required business fields clearly and is less dependent on physical column positions.

# Q4 — Inclusive/Exclusive Trap

The assessment asks you to predict both slices before running them.

The column order is:
0 account_id
1 company
2 signup_ts
3 country
4 seats
5 mrr
6 plan
7 last_active
8 churned

df.loc[:, 'company':'mrr']

Prediction:
company
signup_ts
country
seats
mrr

Total = 5 columns


df.iloc[:, 1:5]

Prediction:
company
signup_ts
country
seats

Total = 4 columns

In [ ]:
# ============================================================
# Q4. INCLUSIVE / EXCLUSIVE TRAP
# ============================================================

loc_result = df.loc[:, 'company':'mrr']

iloc_result = df.iloc[:, 1:5]

print("Columns returned by .loc:")
print(loc_result.columns.tolist())

print("\nNumber of columns returned by .loc:")
print(loc_result.shape[1])

print("\nColumns returned by .iloc:")
print(iloc_result.columns.tolist())

print("\nNumber of columns returned by .iloc:")
print(iloc_result.shape[1])

# Rationale

.loc uses labels and includes both the starting and ending labels in a label slice, so company:mrr includes mrr. .iloc uses integer positions and follows Python's normal exclusive stop rule, so position 5 is not included in 1:5.

# Q5 — Data Quality Diagnosis

This is the main cleaning section. The assessment explicitly says to distinguish real dirt from legitimate data rather than applying generic cleaning rules.

In [ ]:
# First inspect everything

# ============================================================
# Q5. DATA QUALITY DIAGNOSIS
# ============================================================

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate account IDs:")
print(
    df[df.duplicated('account_id', keep=False)]
    .sort_values('account_id')
)

print("\nUnique countries:")
print(df['country'].unique())

print("\nUnique plans:")
print(df['plan'].unique())

print("\nUnique churn values:")
print(df['churned'].unique())

print("\nSeats:")
print(df['seats'].tolist())

print("\nMRR:")
print(df['mrr'].tolist())

print("\nSignup timestamps:")
print(df['signup_ts'].tolist())

print("\nLast active dates:")
print(df['last_active'].tolist())

# Create a Diagnosis Table

# Diagnosis table based on the observed dataset

diagnosis = pd.DataFrame({
    'Problem': [
        'Duplicate A-1001 record',
        'Repeated A-1002 with different seats/MRR',
        'Country uses US / United States / us',
        'Plan uses pro / PRO',
        'Churn uses False / FALSE / No / True',
        'Missing seats for A-1008',
        'Missing last_active for A-1009',
        'Missing company for A-1014',
        'A-1003 has 0 seats and 0 MRR',
        'A-1005 has MRR of -99',
        'A-1010 has MRR of 0',
        'A-1012 has 1,000,000 seats and MRR of 90',
        'A-1013 has last_active in 2099',
        'A-1007 last_active is before signup'
    ],

    'Question_Category': [
        'What is one row?',
        'What is one row?',
        'What should each column be?',
        'What should each column be?',
        'What should each column be?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?',
        'What is missing or lying?'
    ],

    'Decision': [
        'Investigate/remove exact duplicate',
        'Do not automatically remove; investigate update history',
        'Standardize if country-level analysis requires it',
        'Standardize capitalization',
        'Standardize after confirming meanings',
        'Investigate/impute or exclude depending on analysis',
        'Investigate; may be legitimate for churned account',
        'Investigate or retain if account can be identified by ID',
        'Investigate rather than automatically remove',
        'Investigate whether -99 is a code or erroneous MRR',
        'Investigate whether zero represents free/non-paying account',
        'Investigate seat count; likely data problem',
        'Flag as impossible for current context',
        'Flag because activity precedes signup'
    ]
})

display(diagnosis)

# Important legitimate-data example

A-1003 has zero seats and zero MRR. This looks unusual, but it is not automatically dirty: it could represent a legitimate zero-seat/free account or an account whose commercial values have not yet been populated. Therefore, I would investigate the business meaning before changing it.

# Q6 — The Duplicate That Isn't

The assessment specifically points out that two account IDs repeat and asks why drop_duplicates() is not necessarily the correct business solution.

In [ ]:
# ============================================================
# Q6. DUPLICATE THAT ISN'T
# ============================================================

duplicate_accounts = df[
    df.duplicated('account_id', keep=False)
].sort_values('account_id')

display(duplicate_accounts)

In [ ]:
# Test drop_duplicates()

# Proposed solution

df_dropped = df.drop_duplicates()

print("Rows before drop_duplicates():", len(df))
print("Rows after drop_duplicates():", len(df_dropped))

print("\nRepeated account IDs after drop_duplicates():")
display(
    df_dropped[
        df_dropped.duplicated('account_id', keep=False)
    ].sort_values('account_id')
)

# Inspect A-1001

print("A-1001:")
display(df[df['account_id'] == 'A-1001'])

# Inspect A-1002

print("A-1002:")
display(df[df['account_id'] == 'A-1002'])

Drop_duplicates() will remove the identical A-1001 row but will not remove the A-1002 update, because those rows are not completely identical.

In [ ]:
# A safer business-key approach could be:

# Identify duplicate account IDs
account_duplicates = df[df.duplicated(
    subset='account_id',
    keep=False
)].sort_values('account_id')

display(account_duplicates)

# Rationale

Drop_duplicates() compares complete rows, so it removes the identical A-1001 record but does not resolve A-1002, whose two rows have different seats and MRR. Before choosing which A-1002 record to retain, I would need business information confirming whether the later row is a legitimate account update or an error.

Business information required

Confirmation from the source/business system about whether the later A-1002 record is a legitimate account update.

# Q7 — Revenue Per Seat

This is another major question. The assessment specifically warns against simply applying IQR outlier removal.

In [ ]:
# First convert the columns to numbers:

# ============================================================
# Q7. REVENUE PER SEAT
# ============================================================

# Make numeric copies
df_analysis = df.copy()

df_analysis['seats_num'] = pd.to_numeric(
    df_analysis['seats'],
    errors='coerce'
)

df_analysis['mrr_num'] = pd.to_numeric(
    df_analysis['mrr'],
    errors='coerce'
)

# Calculate revenue per seat
df_analysis['revenue_per_seat'] = (
    df_analysis['mrr_num'] /
    df_analysis['seats_num']
)

print("Revenue per seat:")
display(
    df_analysis[
        ['account_id', 'company', 'seats_num',
         'mrr_num', 'revenue_per_seat']
    ]
)

# To inspect suspicious values, we can sort by revenue_per_seat:
display(
    df_analysis[
        ['account_id', 'company', 'seats_num',
         'mrr_num', 'revenue_per_seat']
    ].sort_values('revenue_per_seat')
)

# I can also calculate the median baseline:

baseline = df_analysis['revenue_per_seat'].median()

print("Median revenue per seat:", baseline)

# Rationale

The main baseline is approximately 90 per seat for the ordinary accounts. Suspicious cases should be investigated individually rather than removed using a generic IQR rule.

A-1003: 0 seats and 0 MRR — could be a legitimate free/unactivated account or missing commercial data; verify account status. A-1005: MRR = -99 — could be an invalid value or a special credit/refund/internal code; verify the source definition. A-1010: 20 seats and 0 MRR — could be a free/trial account or missing billing data; check billing status. A-1012: 1,000,000 seats and MRR = 90 — highly likely to be an incorrect seat count; verify against the source system.

A-1003 cannot be resolved from the dataset alone. Billing records or the account-status/source-system record would determine whether its zero values are legitimate or missing/incorrect.

# Q8 — Parsing vs Validation

The assessment wants to distinguish a value that cannot be parsed correctly from a value that parses successfully but is logically impossible.

In [ ]:
# ============================================================
# Q8. DATE PARSING AND VALIDATION
# ============================================================

# Parse signup dates
df_analysis['signup_parsed'] = pd.to_datetime(
    df_analysis['signup_ts'],
    errors='coerce'
)

# Parse last active dates
df_analysis['last_active_parsed'] = pd.to_datetime(
    df_analysis['last_active'],
    errors='coerce'
)

print("Parsed signup dates:")
display(
    df_analysis[
        ['account_id', 'signup_ts', 'signup_parsed']
    ]
)

print("\nParsed last active dates:")
display(
    df_analysis[
        ['account_id', 'last_active',
         'last_active_parsed']
    ]
)

# Inspect the Unix timestamp

# A-1003 contains an epoch timestamp

epoch_value = 1675209600

epoch_date = pd.to_datetime(
    epoch_value,
    unit='s'
)

print("Epoch value:", epoch_value)
print("Converted date:", epoch_date)

# Indentify suspicious signup formats

# Flag signup values that are not in the normal
# YYYY-MM-DD style

signup_format_problem = ~analysis_df['signup_ts'].astype(str).str.match(
    r'^\d{4}-\d{2}-\d{2}'
)

display(
    analysis_df[
        signup_format_problem
    ][
        ['account_id', 'signup_ts']
    ]
)

In [ ]:
# Validation problem 1 - activity before signup

# Convert signup date to datetime
analysis_df['signup_date'] = pd.to_datetime(
    analysis_df['signup_ts'],
    errors='coerce'
)

analysis_df['last_active_date'] = pd.to_datetime(
    analysis_df['last_active'],
    errors='coerce'
)

# Last active should normally not be before signup
activity_before_signup = analysis_df[
    analysis_df['last_active_date'] <
    analysis_df['signup_date']
]

print("Last-active dates before signup:")
display(
    activity_before_signup[
        [
            'account_id',
            'signup_date',
            'last_active_date'
        ]
    ]
)

In [ ]:
# Validation problem 2 - future date

# Define a reasonable analysis cutoff
cutoff_date = pd.Timestamp('2024-03-31')

future_activity = analysis_df[
    analysis_df['last_active_date'] >
    cutoff_date
]

print("Future last-active dates:")
display(
    future_activity[
        [
            'account_id',
            'company',
            'last_active_date'
        ]
    ]
)

# Rationale

A parsing problem concerns whether a value can be interpreted as a date in the expected representation, while validation concerns whether a successfully parsed date makes sense in context. A-1003 requires special handling because its signup value is an epoch timestamp, whereas A-1007 has activity before signup and A-1013 has a valid but contextually impossible 2099 activity date.

# Q9 — The Same Null, Two Fates

The assessment asks for one missing label and one missing measure

In [ ]:
# ============================================================
# Q9. SAME NULL, TWO FATES
# ============================================================

# Find all rows containing missing values

missing_rows = df[
    df.isnull().any(axis=1)
]

display(missing_rows)

# Missing seats

missing_seats = df[
    df['seats'].isnull()
]

print("Missing seats:")
display(missing_seats)

# Missing company

missing_company = df[
    df['company'].isnull()
]

print("Missing company:")
display(missing_company)

# Rationale

Dropping A-1008 could be appropriate for a per-seat revenue analysis because the missing seat count prevents that calculation, but it could unnecessarily remove a valid account from other analyses such as total MRR. Similarly, A-1014's missing company name could make a company-level report difficult to interpret, but the account can still contribute to numerical analysis because its account ID, seats, and MRR are present.

# Q10 — Cleaning Is a Function of the Question

The assessment asks for two cleaning decisions that could reasonably be reversed under a different analytical goal.

In [ ]:
# ============================================================
# Q10. CLEANING DEPENDS ON THE QUESTION
# ============================================================

print("""
DECISION 1
---------

I would flag A-1013's last_active date of 2099-01-01 as invalid
for a current customer-activity analysis.

However, if the purpose of the analysis were to investigate
data-entry errors or system migration problems, I would retain
the value because the abnormal date is useful evidence.


DECISION 2
---------

I would flag A-1005's MRR of -99 as suspicious for a revenue
analysis.

However, if the business confirms that -99 is a legitimate
internal code representing a credit, refund, or special account
status, I would retain the original value rather than replacing
or deleting it.


CONCLUSION
----------

A dataset is not inherently clean or dirty; whether a value is
acceptable depends on the analytical question and the business
rules being applied.

Therefore, cleaning is a decision made in relation to the
purpose of the analysis rather than a permanent property of
the dataset.
""")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("""
SUMMARY OF DATA QUALITY FINDINGS
================================

1. A-1001 is an exact duplicate.
2. A-1002 appears twice with different seats and MRR, suggesting
   a possible account update rather than a simple duplicate.
3. Country values are inconsistent: US, United States and us.
4. Plan values are inconsistently capitalized: pro and PRO.
5. Churn values use several representations: False, FALSE, No
   and True.
6. A-1008 has missing seats.
7. A-1009 has missing last_active.
8. A-1014 has a missing company name.
9. A-1003 has zero seats and zero MRR and requires business
   interpretation rather than automatic deletion.
10. A-1005 has an MRR value of -99 and requires investigation.
11. A-1010 has zero MRR despite having 20 seats.
12. A-1012 has an extremely large seat count of 1,000,000.
13. A-1007 has last_active before signup.
14. A-1013 has a future last_active date of 2099-01-01.

The key lesson is that unusual values should be investigated
rather than automatically removed. The correct cleaning decision
depends on the business question and the meaning of the data.
""")